[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanderlab/IvanderLab-Colab-Suite/blob/main/Applio.ipynb)

# 🎵 Applio RVC Suite - Text to Speech (TTS) & AI Song Cover Generator
### ⚡ Powered by `IAHispano/Applio` | Edge-TTS Kloning Suara | AI Song Cover Generator | Cloudflare Free Tunnel

---

### 🌟 Fitur Utama Applio WebUI:
1. **🗣️ Text to Speech (TTS)**: Ketik teks / naskah apa saja ➔ AI langsung membacakannya dengan suara model RVC (Yandere ASMR, Cyrene, Komakaze Rabu, dll)!
2. **🎤 AI Song Cover (Inference)**: Masukkan lagu MP3 apa saja ➔ Suara penyanyinya otomatis diubah jadi suara karakter anime pilihanmu!
3. **📥 Download Model Instan**: Bisa paste link HuggingFace apa saja langsung di tab Download.

---
### 📋 Panduan 3 Langkah Cepat:
1. Jalankan **CELL 1** (Setup Environment Applio ~1-2 menit)
2. Jalankan **CELL 1.5** (Auto-Download Model Yandere ASMR / Cyrene / Rabu)
3. Jalankan **CELL 2** (Launch WebUI) ➔ Klik link Cloudflare / Gradio!


In [ ]:
# ==============================================================================
# 📦 CELL 1: SETUP APPLIO ENVIRONMENT & DEPENDENCIES (FREE TIER COLAB)
# ==============================================================================
import os
import sys
import codecs
import subprocess
from google.colab import drive

# 1. Parameter Google Drive (Opsional: Simpan Model & Hasil TTS/Cover ke Drive)
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}

print("🚀 [1/3] Menyiapkan environment Applio & dependensi AI...")

if USE_GOOGLE_DRIVE and not os.path.exists('/content/drive'):
    try:
        drive.mount('/content/drive')
        print("💾 Google Drive tersambung!")
    except Exception as e:
        print(f"⚠️ Lewati Google Drive: {e}")

os.chdir("/content")
repo_url = codecs.decode("uggcf://tvguho.pbz/VNUvfcnab/Nccyvb/", "rot_13")
repo_name = codecs.decode("Nccyvb", "rot_13")

# Clone Applio repository jika belum ada
if not os.path.exists(f"/content/{repo_name}"):
    print("📦 Mengunduh Applio Core Engine...")
    subprocess.run(["git", "clone", repo_url], check=True)

os.chdir(f"/content/{repo_name}")

# Install Cloudflare binary & system audio packages
subprocess.run(["apt-get", "-y", "install", "-qq", "aria2", "portaudio19-dev", "ffmpeg"], check=False)
if not os.path.exists("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/content/cloudflared"], check=False)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=False)

print("⚡ Menginstal dependensi Python Applio...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
subprocess.run(["pip", "install", "-q", "resampy", "wget", "edge-tts", "jinja2", "pyngrok", "torchcrepe", "torchfcpe", "noisereduce", "pedalboard", "stftpitchshift", "soxr", "einops"], check=False)

print("🔍 Mengunduh pretrained audio models (HiFi-GAN & RMVPE)...")
subprocess.run(["python", "core.py", "prerequisites", "--models", "--pretraineds-hifigan"], check=False)

print("\n🎉 [OK] Setup Applio Selesai! Lanjutkan jalankan CELL 1.5 untuk download model suara, lalu jalankan CELL 2.")


In [ ]:
# ==============================================================================
# 📥 [OPSIONAL] CELL 1.5: AUTO-DOWNLOAD RVC MODEL (YANDERE, CYRENE, RABU, DLL)
# ==============================================================================
import os
import shutil
import zipfile
import subprocess

# @markdown **Masukkan link model RVC dari HuggingFace (File .zip / .pth):**
MODEL_URL = "https://huggingface.co/akiotai/Animegirlmodels/resolve/main/YandereASMR.zip" #@param {type:"string"}

if MODEL_URL and MODEL_URL.strip():
    raw_name = os.path.basename(MODEL_URL.split("?")[0])
    base_name = os.path.splitext(raw_name)[0]
    
    weights_dir = "/content/Applio/assets/weights"
    logs_dir = f"/content/Applio/logs/{base_name}"
    os.makedirs(weights_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)
    
    temp_zip = f"/content/{raw_name}"
    print(f"📥 Mengunduh model {raw_name} via aria2c...")
    subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", MODEL_URL, "-d", "/content", "-o", raw_name], check=False)
    
    if raw_name.endswith(".zip") and os.path.exists(temp_zip):
        print(f"📦 Mengekstrak file model (.pth dan .index ke logs/{base_name}/)...")
        with zipfile.ZipFile(temp_zip, 'r') as zip_ref:
            for member in zip_ref.namelist():
                fname = os.path.basename(member)
                if not fname:
                    continue
                if fname.endswith(".pth"):
                    # Simpan ke logs/{base_name}/ (Applio scanner) & assets/weights/
                    with zip_ref.open(member) as src, open(os.path.join(logs_dir, fname), "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    shutil.copyfile(os.path.join(logs_dir, fname), os.path.join(weights_dir, fname))
                    print(f"  ✅ File Voice Model: {fname} -> logs/{base_name}/")
                elif fname.endswith(".index"):
                    with zip_ref.open(member) as src, open(os.path.join(logs_dir, fname), "wb") as dst:
                        shutil.copyfileobj(src, dst)
                    print(f"  ✅ File Feature Index: {fname} -> logs/{base_name}/")
        os.remove(temp_zip)
    elif raw_name.endswith(".pth") and os.path.exists(temp_zip):
        shutil.copyfile(temp_zip, os.path.join(logs_dir, raw_name))
        shutil.move(temp_zip, os.path.join(weights_dir, raw_name))
        print(f"  ✅ File Voice Model: {raw_name} -> logs/{base_name}/")
        
    print(f"\n🎉 Model {base_name} berhasil terpasang di Applio dan siap dipakai di TTS / Song Cover!")
else:
    print("ℹ️ Tidak ada URL yang dimasukkan. Anda tetap bisa download lewat tab 'Download' di WebUI Applio!")


In [ ]:
# ==============================================================================
# 🚀 CELL 2: LAUNCH APPLIO WEBUI (TEXT-TO-SPEECH, SONG COVER & INFERENCE)
# ==============================================================================
import os
import re
import glob
import json
import time
import shutil
import socket
import threading
import subprocess

# 🩹 Auto-heal: Otomatis instal dependensi jika belum ada
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
subprocess.run(["pip", "install", "-q", "resampy", "wget", "edge-tts", "jinja2", "pyngrok", "torchcrepe", "torchfcpe", "noisereduce", "pedalboard", "stftpitchshift", "soxr", "einops"], check=False)

# 🔄 Auto-sync: Pastikan semua file model .pth terdeteksi oleh Applio logs
os.makedirs("/content/Applio/logs", exist_ok=True)
for pth in glob.glob("/content/Applio/assets/weights/*.pth") + glob.glob("/content/*.pth"):
    bname = os.path.splitext(os.path.basename(pth))[0]
    target_dir = f"/content/Applio/logs/{bname}"
    os.makedirs(target_dir, exist_ok=True)
    target_pth = os.path.join(target_dir, os.path.basename(pth))
    if not os.path.exists(target_pth):
        shutil.copyfile(pth, target_pth)

METHOD = "Cloudflare (Rekomendasi - Free & Unlimited)" #@param ["Cloudflare (Rekomendasi - Free & Unlimited)", "Gradio Share", "Ngrok"]
NGROK_TOKEN = "" #@param {type:"string"}

PORT = 6969
os.chdir("/content/Applio")

public_url = ""

# Inisialisasi Cloudflare Tunnel
if "Cloudflare" in METHOD:
    print("🌐 [1/2] Membuka jalur Cloudflare Quick Tunnel...")
    p_cf = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    for line in p_cf.stdout:
        if "trycloudflare.com" in line:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if m:
                public_url = m.group(0)
                break

# Background Listener untuk print banner saat server 100% siap
def wait_and_print():
    while True:
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        res = sock.connect_ex(('127.0.0.1', PORT))
        sock.close()
        if res == 0:
            time.sleep(1)
            print("\n" + "="*60)
            print("🎉 APPLIO WEBUI (TTS, SONG COVER & VOICE CHANGER) SIAP!")
            if public_url:
                print(f"👉 LINK CLOUDFLARE : {public_url}")
            print("="*60 + "\n")
            break

threading.Thread(target=wait_and_print, daemon=True).start()

print("🚀 [2/2] Meluncurkan Applio Server...")
if "Gradio" in METHOD:
    !python app.py --server-name 0.0.0.0 --port 6969 --share
elif "Ngrok" in METHOD:
    import ngrok
    ngrok.kill()
    listener = ngrok.forward(6969, authtoken=NGROK_TOKEN)
    print(f"👉 Link Ngrok: {listener.url()}")
    !python app.py --server-name 0.0.0.0 --port 6969
else:
    !python app.py --server-name 0.0.0.0 --port 6969
